<a href="https://colab.research.google.com/github/VeenusDadri/training/blob/master/advance_pyspark/Repartition_and_Coalesce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Spark Training').getOrCreate()
sc = spark.sparkContext

In [3]:
print(os.path.exists("/content/employees.csv"))

True


In [4]:
emp_df=spark.read.csv("/content/employees.csv",header=True,inferSchema=True)
emp_df.printSchema()

root
 |-- Employee ID: integer (nullable = true)
 |-- Store ID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Position: string (nullable = true)



In [5]:
emp_df.count()

404

In [23]:
emp_df.show()

+-----------+--------+--------------------+-----------------+
|Employee ID|Store ID|                Name|         Position|
+-----------+--------+--------------------+-----------------+
|          1|       1|     Stephen Johnson|    Store Manager|
|          2|       1|       Rebecca Myers|Assistant Manager|
|          3|       1|  Katherine Buchanan|          Cashier|
|          4|       1|       Jessica Hicks|      Stock Clerk|
|          5|       1|          Ryan Gross|  Sales Associate|
|          6|       1|     Jeffery Carlson|  Sales Associate|
|          7|       1|      Melissa Wilson|  Sales Associate|
|          8|       1|        Edward Hicks|  Sales Associate|
|          9|       1|        Shari Jordan|  Sales Associate|
|         10|       1|Mrs. Melissa Cald...|  Sales Associate|
|         11|       1|       Regina Nelson|  Sales Associate|
|         12|       1|        Kevin Dawson|  Sales Associate|
|         13|       1|       Nicole Murphy|  Sales Associate|
|       

In [6]:
emp_df.rdd.getNumPartitions()

1

In [7]:
repartioned_df=emp_df.repartition(4)
repartioned_df.rdd.getNumPartitions()

4

In [8]:
from pyspark.sql.functions import*
repartioned_df.withColumn("partitionID",spark_partition_id()).groupBy("partitionID").count().show()

+-----------+-----+
|partitionID|count|
+-----------+-----+
|          0|  101|
|          1|  101|
|          2|  101|
|          3|  101|
+-----------+-----+



In [9]:
repartition_with_column_df=emp_df.repartition(3,col("Name"))
repartition_with_column_df.rdd.getNumPartitions()

3

In [10]:
repartition_with_column_df.withColumn("partitionID",spark_partition_id()).groupBy("partitionID").count().show()

+-----------+-----+
|partitionID|count|
+-----------+-----+
|          0|  133|
|          1|  137|
|          2|  134|
+-----------+-----+



In [11]:
repartition_exceed_df=emp_df.repartition(300,col("Name"))
repartition_exceed_df.rdd.getNumPartitions()

300

In [12]:
repartition_exceed_df.withColumn("partitionID",spark_partition_id()).groupBy("partitionID").count().show()

+-----------+-----+
|partitionID|count|
+-----------+-----+
|          0|    2|
|          1|    1|
|          2|    2|
|          4|    3|
|          5|    2|
|          6|    1|
|          8|    2|
|          9|    3|
|         10|    1|
|         14|    1|
|         15|    1|
|         16|    1|
|         18|    2|
|         19|    2|
|         21|    1|
|         22|    2|
|         23|    4|
|         24|    1|
|         25|    3|
|         26|    2|
+-----------+-----+
only showing top 20 rows



In [13]:
coalesced_df=emp_df.coalesce(4)
coalesced_df.rdd.getNumPartitions()

1

In [14]:
repartioned_df1=emp_df.repartition(8)
repartioned_df1.rdd.getNumPartitions()

8

In [15]:
coalesced_df=repartioned_df1.coalesce(3)
coalesced_df.rdd.getNumPartitions()

3

In [16]:
coalesced_df.withColumn("partitionID",spark_partition_id()).groupBy("partitionID").count().show()

+-----------+-----+
|partitionID|count|
+-----------+-----+
|          0|  101|
|          1|  151|
|          2|  152|
+-----------+-----+



In [17]:
r_df=repartioned_df1.repartition(3)
print(r_df.rdd.getNumPartitions())
r_df.withColumn("partitionID",spark_partition_id()).groupBy("partitionID").count().show()


3
+-----------+-----+
|partitionID|count|
+-----------+-----+
|          0|  135|
|          1|  135|
|          2|  134|
+-----------+-----+



In [18]:
spark.conf.get("spark.sql.shuffle.partitions")

'200'

because repartition by default takes in the value present in spark.sql.shuffle.partitions if integer value is not explicitly provided

In [19]:
r2_df=emp_df.repartition("Name")
r2_df.rdd.getNumPartitions()

1

In [20]:
spark.conf.set("spark.sql.shuffle.partitions", "300")
r2_df=emp_df.repartition("Name")
r2_df.rdd.getNumPartitions()

1

In [21]:
repartioned8_df=emp_df.repartition(8)
repartioned8_df.rdd.getNumPartitions()

8

In [27]:
repartioned8_df.withColumn("partitionID",spark_partition_id()).orderBy("Store ID").show()

+-----------+--------+--------------------+-----------------+-----------+
|Employee ID|Store ID|                Name|         Position|partitionID|
+-----------+--------+--------------------+-----------------+-----------+
|          7|       1|      Melissa Wilson|  Sales Associate|          1|
|          9|       1|        Shari Jordan|  Sales Associate|          7|
|         11|       1|       Regina Nelson|  Sales Associate|          6|
|          2|       1|       Rebecca Myers|Assistant Manager|          7|
|         12|       1|        Kevin Dawson|  Sales Associate|          5|
|          1|       1|     Stephen Johnson|    Store Manager|          4|
|          5|       1|          Ryan Gross|  Sales Associate|          5|
|          4|       1|       Jessica Hicks|      Stock Clerk|          3|
|         10|       1|Mrs. Melissa Cald...|  Sales Associate|          5|
|          6|       1|     Jeffery Carlson|  Sales Associate|          0|
|         13|       1|       Nicole Mu

In [29]:
repartioned8_withcolumn_df=emp_df.repartition(8,"Store ID")
repartioned8_withcolumn_df.rdd.getNumPartitions()

8

In [31]:
r8_df=repartioned8_withcolumn_df.withColumn("partitionID",spark_partition_id())
r8_df.orderBy("Store ID").show()


+-----------+--------+--------------------+-----------------+-----------+
|Employee ID|Store ID|                Name|         Position|partitionID|
+-----------+--------+--------------------+-----------------+-----------+
|          1|       1|     Stephen Johnson|    Store Manager|          3|
|          2|       1|       Rebecca Myers|Assistant Manager|          3|
|          3|       1|  Katherine Buchanan|          Cashier|          3|
|          4|       1|       Jessica Hicks|      Stock Clerk|          3|
|          5|       1|          Ryan Gross|  Sales Associate|          3|
|          6|       1|     Jeffery Carlson|  Sales Associate|          3|
|          7|       1|      Melissa Wilson|  Sales Associate|          3|
|          8|       1|        Edward Hicks|  Sales Associate|          3|
|          9|       1|        Shari Jordan|  Sales Associate|          3|
|         10|       1|Mrs. Melissa Cald...|  Sales Associate|          3|
|         11|       1|       Regina Ne

In [42]:
from pyspark.sql import functions as F
r8_df.groupBy("Store ID").agg(F.collect_list("PartitionID").alias("IDs")).show(40,truncate=False)

+--------+---------------------------------------+
|Store ID|IDs                                    |
+--------+---------------------------------------+
|12      |[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]|
|13      |[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]   |
|14      |[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]      |
|18      |[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]         |
|6       |[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]      |
|9       |[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]   |
|16      |[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]         |
|17      |[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]   |
|23      |[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]      |
|5       |[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]   |
|10      |[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]|
|31      |[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]|
|1       |[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]|
|3       |[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]   |
|7       |[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]|
|11      |[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]   |
|26      |[3, 3, 3, 3, 3, 3, 3,

In [39]:
emp_df.select("Store Id").distinct().count()

35